# Smoke Test - HotpotQA
5 examples, real LLM calls, no plots. Exercises `load_hotpotqa → poison_hotpotqa (1 fact) → qa_scorer`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import yaml
from pathlib import Path

with open("../configs/config.yaml") as f:
    cfg = yaml.safe_load(f)

N_SMOKE     = 5
MODEL       = cfg["models"]["default"]
PROMPT_TYPE = cfg["prompts"].get("default_qa", "standard_qa")
TEMPERATURE = cfg["models"]["temperature"]

print(f"model={MODEL}  prompt={PROMPT_TYPE}  k={cfg['retrieval']['k']}  n_smoke={N_SMOKE}")

In [ ]:
from src.data.download_hotpotqa import download
from src.data.hotpotqa_loader import load_hotpotqa

dev_path = Path("..") / cfg["dataset"]["hotpotqa_dev"]
download(target=dev_path)

examples = load_hotpotqa(str(dev_path), max_examples=N_SMOKE)
print(f"Loaded {len(examples)} examples")
for ex in examples:
    print(f"  {ex['question']!r}  ->  {ex['answer']!r}")

In [ ]:
from src.retrieval.embedder import Embedder
from src.retrieval.retriever import Retriever
from src.generation.llm_client import HuggingFaceClient

emb_cache = os.path.join("../", cfg["cache"]["dir"], cfg["cache"]["embeddings_subdir"])
embedder = Embedder(model_name=cfg["retrieval"]["embedding_model"], cache_dir=emb_cache)
retriever = Retriever(embedder=embedder, k=cfg["retrieval"]["k"])

llm_cache = os.path.join("../", cfg["cache"]["dir"], cfg["cache"]["llm_subdir"])
llm = HuggingFaceClient(model=MODEL, temperature=TEMPERATURE, cache_dir=llm_cache)
print(f"k={cfg['retrieval']['k']}  model={MODEL}")

In [ ]:
from src.data.hotpotqa_loader import load_hotpotqa
from src.evaluation import qa_scorer

poisoned = load_hotpotqa("../" + cfg["dataset"]["hotpotqa_dev_poisoned"], max_examples=N_SMOKE)
print(f"Loaded {len(poisoned)} pre-computed poisoned examples")

with embedder, llm:
    # r=0: original supporting facts
    metrics_clean = qa_scorer.run(
        examples=examples,
        retriever=retriever,
        llm=llm,
        prompt_type=PROMPT_TYPE,
        seed=cfg["seed"],
    )
    # r=1: pre-computed LLM-negated supporting facts (loaded from dev_poisoned.jsonl)
    metrics_poisoned = qa_scorer.run(
        examples=poisoned,
        retriever=retriever,
        llm=llm,
        prompt_type=PROMPT_TYPE,
        seed=cfg["seed"],
    )

print("=== r=0 (clean) ===")
for k, v in metrics_clean.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

print("\n=== r=1 (poisoned) ===")
for k, v in metrics_poisoned.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

print("\nHotpotQA smoke passed." if metrics_clean and metrics_poisoned else "Check errors above.")